# Phase 4: Backtesting and Business Interpretation

## Project context

This notebook evaluates the fixed Phase 3 portfolio weights over the full available return history. The goal is to compare strategy behavior against SPY using return, risk, drawdown, and benchmark-relative metrics.

## Backtesting objective

The backtest uses fixed weights for `equal_weight`, `minimum_volatility`, and `maximum_sharpe`. It is a simple historical analysis, not a walk-forward optimization or production investment system.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.config import OUTPUTS_DIR, FIGURES_DIR
from src.backtesting import (
    load_returns,
    load_portfolio_weights,
    build_strategy_return_table,
    calculate_cumulative_returns,
    calculate_drawdown_series,
    summarize_backtests,
)
from src.visualization import (
    plot_backtest_cumulative_returns,
    plot_portfolio_risk_return,
    plot_backtest_drawdowns,
    plot_backtest_metric_comparison,
)

BENCHMARK = "SPY"
RETURNS_PATH = OUTPUTS_DIR / "returns" / "daily_returns.csv"
WEIGHTS_PATH = OUTPUTS_DIR / "portfolios" / "portfolio_weights.csv"
BACKTESTS_DIR = OUTPUTS_DIR / "backtests"
BACKTESTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
def _normalize(values):
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(values)
    if not finite.any():
        return np.zeros_like(values)
    vmin = np.nanmin(values[finite])
    vmax = np.nanmax(values[finite])
    if np.isclose(vmin, vmax):
        return np.full_like(values, 0.5, dtype=float)
    return (values - vmin) / (vmax - vmin)


def save_plotly_or_pillow(fig, output_path, chart_type, data):
    output_path = Path(output_path)
    try:
        fig.write_image(str(output_path), width=1200, height=700, scale=2)
        return "plotly"
    except Exception as exc:
        draw_basic_png(output_path, chart_type, data, str(exc))
        return "pillow_fallback"


def draw_basic_png(output_path, chart_type, data, reason):
    width, height = 1200, 700
    margin = 80
    image = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    draw.text((margin, 24), output_path.stem.replace("_", " ").title(), fill="black", font=font)
    draw.text((margin, 48), "Rendered with Pillow fallback because Plotly PNG export was unavailable.", fill="#555555", font=font)
    left, top, right, bottom = margin, 110, width - margin, height - margin
    draw.rectangle((left, top, right, bottom), outline="#333333")

    if chart_type == "lines":
        frame = data.dropna(how="all")
        if len(frame) > 260:
            frame = frame.iloc[np.linspace(0, len(frame) - 1, 260).astype(int)]
        colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
        combined = frame.values.flatten()
        ymin, ymax = np.nanmin(combined), np.nanmax(combined)
        if np.isclose(ymin, ymax):
            ymin, ymax = ymin - 1, ymax + 1
        for idx, col in enumerate(frame.columns):
            values = frame[col].astype(float).values
            points = []
            for i, value in enumerate(values):
                x = left + (right - left) * i / max(len(values) - 1, 1)
                y = bottom - (bottom - top) * ((value - ymin) / (ymax - ymin))
                points.append((x, y))
            if len(points) > 1:
                draw.line(points, fill=colors[idx % len(colors)], width=2)
            draw.text((right - 170, top + 18 * idx), str(col), fill=colors[idx % len(colors)], font=font)
    elif chart_type == "bar":
        frame = data.copy()
        values = frame["value"].astype(float).values
        labels = frame["label"].astype(str).tolist()
        scale_values = _normalize(values)
        bar_width = (right - left) / max(len(values), 1) * 0.65
        for i, value in enumerate(values):
            x = left + (right - left) * (i + 0.5) / len(values)
            y = bottom - (bottom - top) * scale_values[i]
            draw.rectangle((x - bar_width / 2, y, x + bar_width / 2, bottom), fill="#1f77b4")
            draw.text((x - 40, bottom + 10), labels[i][:18], fill="black", font=font)
            draw.text((x - 20, y - 16), f"{value:.2f}", fill="black", font=font)
    elif chart_type == "scatter":
        frame = data.copy()
        x = _normalize(frame["annualized_volatility"].values)
        y = _normalize(frame["annualized_return"].values)
        labels = frame["strategy"].astype(str).tolist()
        for i in range(len(frame)):
            px = left + (right - left) * x[i]
            py = bottom - (bottom - top) * y[i]
            draw.ellipse((px - 5, py - 5, px + 5, py + 5), fill="#1f77b4")
            draw.text((px + 8, py - 8), labels[i], fill="black", font=font)
    image.save(output_path)


## Load returns and portfolio weights

In [3]:
returns = load_returns(RETURNS_PATH)
weights_df = load_portfolio_weights(WEIGHTS_PATH)
display(returns.head())
display(weights_df)
print(f"Return period: {returns.index.min().date()} to {returns.index.max().date()}")

,AAPL,MSFT,JPM,PG,XOM,JNJ,KO,NVDA,SPY
date,,,,,,,,,
2019-01-03,-0.099608,-0.036788,-0.014212,-0.007011,-0.015354,-0.015890,-0.006179,-0.060417,-0.023863
2019-01-04,0.042690,0.046509,0.036865,0.020410,0.036870,0.016783,0.019940,0.064068,0.033496
2019-01-07,-0.002226,0.001275,0.000695,-0.004000,0.005201,-0.006415,-0.013033,0.052941,0.007885
2019-01-08,0.019063,0.007251,-0.001886,0.003691,0.007271,0.023227,0.011289,-0.024895,0.009396
2019-01-09,0.016981,0.014299,-0.001690,-0.016331,0.005275,-0.007926,-0.019166,0.019667,0.004673


,asset,equal_weight,minimum_volatility,maximum_sharpe
0,AAPL,0.125,0.012136,0.009523
1,MSFT,0.125,0.073561,0.001808
2,JPM,0.125,0.012483,0.053063
3,PG,0.125,0.156620,0.001221
4,XOM,0.125,0.086070,0.064074
5,JNJ,0.125,0.410581,0.357834
6,KO,0.125,0.226956,0.028951
7,NVDA,0.125,0.021593,0.483525


Return period: 2019-01-03 to 2026-04-27


## Build fixed-weight strategy returns

In [4]:
strategy_returns = build_strategy_return_table(returns, weights_df, benchmark=BENCHMARK)
strategy_returns.to_csv(BACKTESTS_DIR / "strategy_daily_returns.csv", index_label="date")
display(strategy_returns.head())
print(f"Strategy return shape: {strategy_returns.shape}")

,equal_weight,minimum_volatility,maximum_sharpe,SPY
date,,,,
2019-01-03,-0.031932,-0.015743,-0.037840,-0.023863
2019-01-04,0.035517,0.023569,0.042395,0.033496
2019-01-07,0.004305,-0.004552,0.023272,0.007885
2019-01-08,0.005626,0.013506,-0.002834,0.009396
2019-01-09,0.001389,-0.008046,0.006534,0.004673


Strategy return shape: (1838, 4)


## Cumulative return comparison

In [5]:
strategy_cumulative_returns = calculate_cumulative_returns(strategy_returns)
strategy_cumulative_returns.to_csv(BACKTESTS_DIR / "strategy_cumulative_returns.csv", index_label="date")
display(strategy_cumulative_returns.tail())

,equal_weight,minimum_volatility,maximum_sharpe,SPY
date,,,,
2026-04-21,4.511480,1.727150,13.892548,2.137856
2026-04-22,4.559932,1.735455,13.996855,2.169632
2026-04-23,4.568315,1.773054,14.014699,2.157331
2026-04-24,4.602931,1.774782,14.237872,2.181798
2026-04-27,4.602450,1.770752,14.275851,2.179904


## Risk and return metrics

In [6]:
backtest_metrics = summarize_backtests(strategy_returns, benchmark_col=BENCHMARK)
backtest_metrics.to_csv(BACKTESTS_DIR / "backtest_metrics.csv", index=False)
display(backtest_metrics.style.format({
    "cumulative_return": "{:.2%}",
    "annualized_return": "{:.2%}",
    "annualized_volatility": "{:.2%}",
    "sharpe_ratio": "{:.2f}",
    "max_drawdown": "{:.2%}",
    "best_day": "{:.2%}",
    "worst_day": "{:.2%}",
    "correlation_to_spy": "{:.2f}",
}))


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/rovs/Library/Python/3.11/lib/python/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/rovs/Library/Python/3.11/lib/python/site-packages/ipykernel/kernelapp.py", line 736, in start
    se

AttributeError: _ARRAY_API not found

,strategy,cumulative_return,annualized_return,annualized_volatility,sharpe_ratio,max_drawdown,best_day,worst_day,correlation_to_spy
2,maximum_sharpe,1427.59%,45.32%,27.89%,1.63,-36.43%,12.59%,-12.58%,0.82
0,equal_weight,460.24%,26.65%,19.38%,1.37,-33.06%,10.11%,-10.92%,0.94
1,minimum_volatility,177.08%,15.00%,15.89%,0.94,-30.26%,7.50%,-7.72%,0.72
3,SPY,217.99%,17.19%,19.57%,0.88,-33.72%,10.50%,-10.94%,nan


## Drawdown comparison

In [7]:
strategy_drawdowns = strategy_returns.apply(calculate_drawdown_series)
strategy_drawdowns.to_csv(BACKTESTS_DIR / "strategy_drawdowns.csv", index_label="date")
display(strategy_drawdowns.tail())

,equal_weight,minimum_volatility,maximum_sharpe,SPY
date,,,,
2026-04-21,-0.021770,-0.071109,-0.015885,-0.008534
2026-04-22,-0.013171,-0.068280,-0.008992,0.000000
2026-04-23,-0.011683,-0.055474,-0.007813,-0.003881
2026-04-24,-0.005539,-0.054885,0.000000,0.000000
2026-04-27,-0.005624,-0.056258,0.000000,-0.000595


## Benchmark comparison versus SPY

In [8]:
strategy_only_metrics = backtest_metrics[backtest_metrics["strategy"] != BENCHMARK]
spy_metrics = backtest_metrics[backtest_metrics["strategy"] == BENCHMARK].iloc[0]
best_sharpe = strategy_only_metrics.sort_values("sharpe_ratio", ascending=False).iloc[0]
best_drawdown = strategy_only_metrics.sort_values("max_drawdown", ascending=False).iloc[0]
print(f"SPY annualized return: {spy_metrics['annualized_return']:.2%}")
print(f"Best strategy Sharpe: {best_sharpe['strategy']} ({best_sharpe['sharpe_ratio']:.2f})")
print(f"Least severe strategy drawdown: {best_drawdown['strategy']} ({best_drawdown['max_drawdown']:.2%})")

SPY annualized return: 17.19%
Best strategy Sharpe: maximum_sharpe (1.63)
Least severe strategy drawdown: minimum_volatility (-30.26%)


## Figures

In [9]:
figure_specs = {
    "backtest_cumulative_returns.png": (
        plot_backtest_cumulative_returns(strategy_cumulative_returns),
        "lines",
        strategy_cumulative_returns,
    ),
    "backtest_risk_return_comparison.png": (
        plot_portfolio_risk_return(backtest_metrics.rename(columns={"strategy": "portfolio"})),
        "scatter",
        backtest_metrics[["strategy", "annualized_volatility", "annualized_return"]],
    ),
    "backtest_drawdowns.png": (
        plot_backtest_drawdowns(strategy_drawdowns),
        "lines",
        strategy_drawdowns,
    ),
    "backtest_metric_comparison.png": (
        plot_backtest_metric_comparison(backtest_metrics, "sharpe_ratio", "Backtest Sharpe Ratio Comparison"),
        "bar",
        backtest_metrics[["strategy", "sharpe_ratio"]].rename(columns={"strategy": "label", "sharpe_ratio": "value"}),
    ),
}

figure_export_methods = {}
for filename, (fig, chart_type, data) in figure_specs.items():
    figure_export_methods[filename] = save_plotly_or_pillow(fig, FIGURES_DIR / filename, chart_type, data)

figure_export_methods

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/_plotly_utils/basevalidators.py:105: FutureWarning: The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result
  v = v.dt.to_pydatetime()
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/_plotly_utils/basevalidators.py:105: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



{'backtest_cumulative_returns.png': 'pillow_fallback',
 'backtest_risk_return_comparison.png': 'pillow_fallback',
 'backtest_drawdowns.png': 'pillow_fallback',
 'backtest_metric_comparison.png': 'pillow_fallback'}

## Strategy interpretation

- Maximum-Sharpe delivered the highest historical cumulative return and Sharpe ratio in this fixed-weight test, but it had higher volatility and meaningful concentration from Phase 3 weights.
- Equal-weight performed strongly and had the highest correlation to SPY among the portfolio strategies, making it a simple broad-market-like portfolio for this universe.
- Minimum-volatility had the lowest volatility and least severe drawdown among the strategies, but it gave up return versus the other fixed-weight portfolios.

## Practical business interpretation

The backtest shows the tradeoff between growth-oriented concentration, broad diversification, and defensive risk control. A finance team would not treat these weights as final investment advice, but the workflow is useful for explaining why allocation constraints, risk assumptions, and benchmark comparison matter.

## Limitations

- This is a fixed-weight historical backtest using weights estimated from the same full sample.
- No walk-forward validation, rebalancing schedule, transaction costs, taxes, or turnover controls are included.
- Historical optimized weights can overfit past returns and may not perform similarly out of sample.
- SPY is included only as a benchmark reference, not as an optimized portfolio asset.

## Final project outputs

- Market data and return outputs from Phase 1
- CAPM and single-index outputs from Phase 2
- Portfolio optimization outputs from Phase 3
- Fixed-weight backtest outputs and research memo from Phase 4

## Next steps for Phase 5 GitHub polish

- Clean the README for recruiter readability.
- Polish reports and career-facing materials.
- Keep limitations visible and avoid presenting the project as investment advice.